In [1]:
%pip install datasets
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [49]:
# Initial setup
from datasets import load_dataset
import dotenv
import os
import requests
import pprint
from datetime import datetime
import csv

dotenv.load_dotenv()
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
ALINIA_STAGING_API_KEY = os.getenv("ALINIA_STAGING_API_KEY")

# Helper Classes

In [50]:
class APIException(Exception):
    def __init__(self, input_json: dict, status_code: int, response_text: str):
        self.input_json = input_json
        self.status_code = status_code
        self.response_text = response_text
    
    def __str__(self) -> str:
        return f"API Error:\nstatus code: {self.status_code}\nresponse text: {self.response_text}\ninput json:{self.input_json}"

class Results:
    def __init__(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        self.accuracy = 0
        self.precision = 0
        self.recall = 0
        self. f1_score = 0
    
    def calculate_stats(self) -> None:
        if self.tp > 0 and self.fp > 0:
            self.accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn)
        else:
            self.accuracy = None

        if self.tp > 0:
            self.precision = self.tp / (self.tp + self.fp)
            self.recall = self.tp / (self.tp + self.fn)
        else:
            self.precision = None
            self.recall = None

        if self.precision is not None and self.recall is not None:
            self.f1_score = 2 * (self.precision * self.recall) / (self.precision + self.recall)
        else:
            self.f1_score = None

    def __str__(self) -> str:
        return f"tp: {self.tp}" + "\n" + f"tn: {self.tn}" + "\n" + f"fp: {self.fp}"+ "\n" + f"fn: {self.fn}"+ "\n" + f"accuracy: {self.accuracy}"+ "\n" + f"precision: {self.precision}"+ "\n" + f"recall: {self.recall}"+ "\n" + f"f1_score: {self.f1_score}"

# Helper Functions

In [51]:
# Helper functions
def get_detection_config_json(
        violence: bool = False, 
        hate: bool = False, 
        wrongdoing: bool = False,
        sexual: bool = False) -> dict:
    
    detection_config_json = {}

    # Set the safety guard flags
    if violence or hate or wrongdoing or sexual:
        detection_config_json["safety"] = {}

        if violence: detection_config_json["safety"]["violence"] = True
        if hate:  detection_config_json["safety"]["hate"] = True
        if wrongdoing:  detection_config_json["safety"]["wrongdoing"] = True
        if sexual:  detection_config_json["safety"]["sexual"] = True
    
    return detection_config_json

def get_alinia_input_json(input_str: str, detection_config: dict) -> dict:
    return {
        "input": input_str,
        "detection_config": detection_config
    }

def evaluate(input_json : dict) -> dict:
    response = requests.post("https://staging.api.alinia.ai/moderations/",
        headers = {"Authorization": f"Bearer {ALINIA_STAGING_API_KEY}"},
        json = input_json,
    )
    if not response.ok:
        print(f">>>> HTTP error {response.status_code}: {response.text}")
        raise APIException(input_json, response.status_code, response.text)
    try:
        return response.json()
    except requests.exceptions.JSONDecodeError:
        print(f">>>> Failed to decode JSON response (status {response.status_code}): {response.text!r}")
        raise APIException(input_json, response.status_code, response.text)
    
def evaluate_example(example, text_parameter, class_parameter, positive_class, negative_class, detection_config, results_obj, error_path):
    input_str = example[text_parameter]
    label = example[class_parameter]
    response_json = None

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    try:
        response_json = evaluate(input_json)
    except APIException as e:
        print(e)
        with open(error_path, "a") as errors_fp:
            errors_fp.write(str(datetime.now()) + "\n" + str(e) + "\n")
    
    is_flagged = False

    if response_json is not None and 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']

    if response_json is not None:
        if is_flagged and label == positive_class:
            results_obj.tp += 1
        elif is_flagged and label == negative_class:
            results_obj.fp += 1
        elif not is_flagged and label == negative_class:
            results_obj.tn += 1
        elif not is_flagged and label == positive_class:
            results_obj.fn += 1


In [52]:
# Create configs
HATE_CONFIG = get_detection_config_json(hate = True)
SEXUAL_CONFIG = get_detection_config_json(sexual = True)
VIOLENCE_CONFIG = get_detection_config_json(violence = True)
WRONGDOING_CONFIG = get_detection_config_json(wrongdoing = True)

# Hate Speech Evaluations

In [53]:
# Hate Speech Public Evaluations

# Doesn't use general version since we're populating 2 results per call.
def evaluate_hate_1(example, detection_config, results, results2):
    input_str = example["tweet"]
    label = example["class"]

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    response_json = evaluate(input_json)
    
    errors = []
    is_flagged = False

    if 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']
        else:
            errors.append(f'>>>> Missing result, flagged for: {example}')
            print(f'>>>> Missing result.flagged for: {example}')
    else:
        errors.append(f'>>>> Missing result for: {example}')
        print(f'>>>> Missing result for: {example}')

    # pprint.pp(response_json)
    # print(f"is_flagged: {is_flagged}, label: {label}")

    # Only Hate class as positive label
    if is_flagged and label == 0:
        results.tp += 1
    elif is_flagged and label > 0:
        results.fp += 1
    elif not is_flagged and label > 0:
        results.tn += 1
    elif not is_flagged and label == 0:
        results.fn += 1

    # Either Hate or Offensive class as positive label
    if is_flagged and (label == 0 or label == 1):
        results2.tp += 1
    elif is_flagged and label == 2:
        results2.fp += 1
    elif not is_flagged and label == 2:
        results2.tn += 1
    elif not is_flagged and (label == 0 or label == 1):
        results2.fn += 1

In [54]:
# ===== Hate Evaluation 1 =====

# Dataset details
# id: id
# count: number of annotators
# hate_speech_count: number of annotators who judged the tweet to be hate speech
# offensive_language_count: number of annotators who judged the tweet to be offensive
# neither_count: number of users who judged the tweet to be neither offensive nor non-offensive
# class: class label for majority of annotators (0: 'hate-speech', 1: 'offensive-language' or 2: 'neither')
# tweet: The content of the tweet
dataset_hate_1 = load_dataset("tdavidson/hate_speech_offensive", token = HUGGINGFACE_API_KEY)
results_hate_1a = Results()
results_hate_1b = Results()

print("Evaluating tdavidson/hate_speech_offensive...")
# small_ds.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1, results_hate_2))
# dataset_hate_1.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1, results_hate_2))
dataset_hate_1.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1a, results_hate_1b))

results_hate_1a.calculate_stats()
print("Results (Hate Label Only):")
print(results_hate_1a)

results_hate_1b.calculate_stats()
print("Results (Hate + Offensive Language Labels):")
print(results_hate_1b)

# Save output to file
with open("./results/results-hate-1.txt", "w") as results_fp:
    results_fp.write("Results (Hate Label Only):\n")
    results_fp.write(str(results_hate_1a))
    results_fp.write("\n\nResults (Hate + Offensive Language Labels):\n")
    results_fp.write(str(results_hate_1b))

Evaluating tdavidson/hate_speech_offensive...


Map:   0%|          | 0/24783 [00:00<?, ? examples/s]

Evaluating: !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...


Map:   0%|          | 2/24783 [00:00<1:45:53,  3.90 examples/s]

Evaluating: !!!!! RT @mleew17: boy dats cold...tyga dwn bad for cuffin dat hoe in the 1st place!!
Evaluating: !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby4life: You ever fuck a bitch and she start to cry? You be confused as shit


...

Evaluating: " cancel that bitch like Nino "
Evaluating: " cant you see these hoes wont change "


Map:   0%|          | 24/24783 [00:04<1:00:43,  6.79 examples/s]

Evaluating: " fuck no that bitch dont even suck dick " &#128514;&#128514;&#128514; the Kermit videos bout to fuck IG up
Evaluating: " got ya bitch tip toeing on my hardwood floors " &#128514; http://t.co/cOU2WQ5L4q


Map:   0%|          | 24/24783 [00:04<1:10:55,  5.82 examples/s]


KeyboardInterrupt: 

In [ ]:
# ===== Hate Evaluation 2 =====
# Dataset details
# (unnamed): row id
# text: the content text
# label: class label (0: non-hate, 1: hate)

dataset_hate_2_en = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "English_test", token = HUGGINGFACE_API_KEY)
dataset_hate_2_fr = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "French_test", token = HUGGINGFACE_API_KEY)
dataset_hate_2_es = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "Spain_test", token = HUGGINGFACE_API_KEY)

results_hate_2_en = Results()
results_hate_2_fr = Results()
results_hate_2_es = Results()

# Test English examples
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (English)...")
# small_ds = dataset_hate_2_en["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_en, "./errors/errors-hate-2.txt"))
dataset_hate_2_en.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_en, "./errors/errors-hate-2.txt"))

results_hate_2_en.calculate_stats()
print("Results Multilingual Hate Speech (English):")
print(results_hate_2_en)

# Save output to file
with open("./results/results-hate-2-en.txt", "w") as results_fp:
    results_fp.write("Results Multilingual Hate Speech (English):\n")
    results_fp.write(str(results_hate_2_en))

# Test French examples
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (French)...")
# small_ds = dataset_hate_2_fr["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_fr, "./errors/errors-hate-2.txt"))
dataset_hate_2_fr.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_fr, "./errors/errors-hate-2.txt"))

results_hate_2_fr.calculate_stats()
print("Results Multilingual Hate Speech (French):")
print(results_hate_2_fr)

# Save output to file
with open("./results/results-hate-2-fr.txt", "w") as results_fp:
    results_fp.write("Results Multilingual Hate Speech (French):\n")
    results_fp.write(str(results_hate_2_fr))

# Test Spanish examples
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (Spanish)...")
# small_ds = dataset_hate_2_es["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_es, "./errors/errors-hate-2.txt"))
dataset_hate_2_es.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_es, "./errors/errors-hate-2.txt"))

results_hate_2_es.calculate_stats()
print("Results Multilingual Hate Speech (Spanish):")
print(results_hate_2_es)

# Save output to file
with open("./results/results-hate-2-es.txt", "w") as results_fp:
    results_fp.write("Results Multilingual Hate Speech (Spanish):\n")
    results_fp.write(str(results_hate_2_es))

Evaluating FrancophonIA/multilingual-hatespeech-dataset (English)...


Map:   0%|          | 1/5906 [00:00<14:51,  6.62 examples/s]

Evaluating: RT @warreports: 8 Pakistanis of #IRGC 's Zeynabiun brigade killed in north Syria, buried today in Qom, Iran. https://twitter.com/warreports/status/700017026358767616/photo/1 
Evaluating: gronk tide pod commercial retarded 


Map:   0%|          | 3/5906 [00:00<15:11,  6.47 examples/s]

Evaluating: @sfrantzman @pmakela1 what they found in Syria are a lot of weapons from WWII in gun shops. And ISIS has a lot of Iranian cache.
Evaluating: After Pakistani Taliban announcement of their support for ISIS, is on the line to pay her prostrated obeisances to ISIS.


...

Map:  41%|████      | 2408/5906 [05:47<08:11,  7.12 examples/s]

Evaluating: @user shit watch twitter get locked saying faggot word #victoryroyale
Evaluating: the homies was a part of ISIS ?, ok.


Map:  41%|████      | 2410/5906 [05:47<08:18,  7.01 examples/s]

Evaluating: Is Pikachu in ISIS?  https://twitter.com/USMC/status/752602019341361152
Evaluating: And now Baghdad. during the holy month of Ramadan      http://fb.me/1nXGsTTDm


Map:  41%|████      | 2412/5906 [05:48<08:17,  7.02 examples/s]

# Violence Guard

In [ ]:
# ===== Violence Evaluation 1 =====

results_violence_1_es = Results()

with open("./datasets/villanos.tsv", "r") as dataset_violence_1_fp:
    reader = csv.DictReader(dataset_violence_1_fp, delimiter="\t")
    for example in reader:
        evaluate_example(example, "text", "label", "VIOLENTO", "NOVIOLENTO", VIOLENCE_CONFIG, results_violence_1_es, "./errors/errors-violence-es.txt")

results_violence_1_es.calculate_stats()
print("Results Violent Speech - VILLANOS (Spanish):")
print(results_violence_1_es)

# Save output to file
with open("./results/results-violence-1-es.txt", "w") as results_fp:
    results_fp.write("Results Violent Speech - VILLANOS (Spanish):\n")
    results_fp.write(str(results_violence_1_es))

{'': '0', 'id': 'tweet85_1234943272562446336', 'text': '@laSextaTV @cayetanaAT @cayetanaAT Grande Cayetana, somos muchos los que pensamos igual que usted sobre la SECTA, con independencia del ideario poltico. Tiene usted ms coj...s que todos los barones del PP juntos.', 'label': 'VIOLENTO', 'insulto': 'SI', 'grado': 'MODERADO', 'rol': 'EJECUTOR', 'tipo': 'OTRO', 'insultos': "['la SECTA', 'Tiene usted ms coj...s que todos los barones del PP juntos']"}
{'': '1', 'id': 'tweet76_1235118763579604993', 'text': '@laSextaTV La Sextapo.', 'label': 'VIOLENTO', 'insulto': 'SI', 'grado': 'MODERADO', 'rol': 'EJECUTOR', 'tipo': 'OTRO', 'insultos': "['La Sextapo']"}
{'': '2', 'id': 'tweet69_1235152882988650496', 'text': '@laSextaTV Retractarse de la verdad? Venga ya', 'label': 'NOVIOLENTO', 'insulto': 'NO', 'grado': '', 'rol': '', 'tipo': '', 'insultos': '[]'}
{'': '3', 'id': 'tweet8_991352279679819776', 'text': 'La #Poltica No es de #Personas es de #Modelos Duque y Vargas son d un Modelo q Aunque Im

# Debugging Cruft

In [45]:
dataset_hate_2_es

DatasetDict({
    test: Dataset({
        features: ['Unnamed: 0', 'text', 'label'],
        num_rows: 1243
    })
})